In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# 🚀 Further Reading: The Papers Behind the KV Cache

*Part of the Vizuara series*
*Estimated time: 20–30 minutes*

## 1. Why Does This Matter?

Welcome back! Throughout this series we have built up a complete picture of the KV cache — from the roofline model, through attention mechanics, through the memory pressure that caching creates. Now we want to do something slightly different.

**We want to hand you the keys to the primary literature.**

Every idea we have discussed traces back to a handful of landmark papers. Reading the originals is not just an academic exercise. It is how you develop the ability to evaluate new claims, spot the assumptions hidden inside a technique, and eventually contribute your own ideas. The gap between "I understand the concept" and "I can read a research paper critically" is smaller than you think — and in this notebook, we will close it.

Here is what we will cover:

| Paper | Year | Core Contribution |
|---|---|---|
| Vaswani et al., *Attention Is All You Need* | 2017 | The transformer and multi-head attention |
| Ainslie et al., *GQA* | 2023 | Grouped Query Attention — practical KV cache reduction |
| Dao et al., *FlashAttention* | 2022 | IO-aware exact attention — beating the memory bottleneck |

By the end of this notebook you will have:

1. **Re-derived** the attention equation from scratch so the Vaswani notation feels like home.
2. **Visualized** the memory savings of GQA versus MHA versus MQA.
3. **Simulated** the IO cost argument that motivates FlashAttention.
4. **Completed** three guided TODOs that test your understanding at each level.

Let us get started.

## 2. Building Intuition

Before we touch a single line of code, let us build the mental scaffolding you need to read these papers with confidence.

### Paper 1 — *Attention Is All You Need* (Vaswani et al., 2017)

Imagine you are in a library with 2,000 books, and you need to answer a question. One approach: read every book front-to-back every time someone asks. That is sequential computation — slow and wasteful. The transformer's proposal was radical: **let every token vote on how relevant every other token is, all at once, in parallel.** Those votes are the attention weights.

The paper introduced the now-ubiquitous **Queries, Keys, and Values** abstraction:

- **Query**: "What am I looking for?"
- **Key**: "What do I contain?"
- **Value**: "What should I contribute if I am selected?"

This is exactly the same structure as a database lookup — except instead of hard matching, we do soft, differentiable matching. And because it is differentiable, gradients can flow and the model can learn what to look for.

**Multi-head attention** then runs this lookup in parallel across many independent "subspaces", letting the model simultaneously attend to syntax, semantics, coreference, and anything else it discovers is useful.

### Paper 2 — *GQA* (Ainslie et al., 2023)

Multi-head attention is wonderful but expensive. Each head maintains its own K and V matrices, and during generation, all of those must be cached and loaded from memory. The KV cache for a single forward pass grows as:

```
2 × num_heads × seq_len × head_dim × bytes_per_element
```

For a 70-billion-parameter model with 64 heads and a 4,096-token context this is already gigabytes. **Multi-Query Attention (MQA)** collapsed all heads down to a single shared K and V — massive memory savings, but a noticeable quality drop. GQA split the difference: group the heads, share K and V within each group, and recover most of the quality while keeping most of the savings. Think of it as carpooling — not everyone shares one car, but small groups share, and the highway is much less congested.

### Paper 3 — *FlashAttention* (Dao et al., 2022)

This one requires the roofline model we built earlier. Standard attention materializes the full N×N attention matrix in VRAM. For a sequence of length N, that is N² elements written and read. FlashAttention's insight: **you never actually need the full matrix in memory at once.** By tiling the computation and keeping intermediate results in the chip's fast on-chip SRAM (instead of slow VRAM), FlashAttention achieves the exact same mathematical result with far fewer trips across the memory bus. It is an **IO-aware** algorithm — it optimizes for memory traffic, not just FLOPs.

### 🤔 Think About This

Here is a question to hold in your mind as we go through the code:

> If GQA reduces the number of K/V heads but keeps the number of Q heads the same, what happens to the arithmetic intensity of the attention computation? Does it go up, down, or stay the same — and does that help or hurt us on the roofline?

We will answer this precisely at the end of the notebook.

## 3. The Mathematics

Let us re-derive everything from first principles so the notation in the papers feels natural.

### 3.1 Scaled Dot-Product Attention (Vaswani et al., Equation 1)

The core attention formula is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

**What this equation says, step by step:**

- $Q \in \mathbb{R}^{n \times d_k}$ — the queries matrix. Each row is one token asking a question.
- $K \in \mathbb{R}^{m \times d_k}$ — the keys matrix. Each row is one token announcing what it contains.
- $QK^T \in \mathbb{R}^{n \times m}$ — the raw attention scores. Entry $(i, j)$ measures how compatible query $i$ is with key $j$. **Computationally:** this is a matrix multiplication costing $O(n \cdot m \cdot d_k)$ FLOPs.
- $/ \sqrt{d_k}$ — the scaling factor. Without it, large $d_k$ makes dot products grow in magnitude and pushes the softmax into regions of near-zero gradient. **Computationally:** one elementwise division, negligible cost.
- $\text{softmax}(\cdot) \in \mathbb{R}^{n \times m}$ — each row of scores becomes a probability distribution over the $m$ keys. **Computationally:** exponentiate, sum, divide — $O(n \cdot m)$.
- $\cdot\, V \in \mathbb{R}^{n \times d_v}$ — take a weighted combination of value vectors. **Computationally:** another matrix multiply, $O(n \cdot m \cdot d_v)$.

### 3.2 Multi-Head Attention

$$\text{MHA}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\, W^O$$

$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

**What this says:** project Q, K, V into $h$ different lower-dimensional spaces and run independent attention in each. Concatenate all outputs and project back. **Computationally:** $h$ attention operations each on $d_k = d_{\text{model}}/h$-dimensional vectors, so total FLOPs are the same as one big attention — but with richer representational capacity.

### 3.3 GQA — The Memory Reduction Formula

In GQA with $G$ groups, $h$ query heads, and $h/G$ KV heads:

$$\text{KV cache size} = 2 \times \frac{h}{G} \times L \times d_k \times \text{bytes}$$

When $G = 1$: standard MHA. When $G = h$: MQA (one KV head). When $1 < G < h$: GQA.

**Computationally:** reducing $G$ from 1 to 8 on a 64-head model shrinks the KV cache by $8\times$, meaning each generation step needs to transfer $8\times$ fewer bytes — moving us rightward on the roofline.

### 3.4 FlashAttention's IO Complexity

Standard attention reads/writes the $N \times N$ score matrix:

$$\text{IO}_{\text{standard}} = O(N^2)$$

FlashAttention tiles the computation into blocks of size $B$:

$$\text{IO}_{\text{flash}} = O\!\left(\frac{N^2 \cdot d}{M}\right)$$

where $M$ is the size of on-chip SRAM. Since $M \gg d$ in practice, $\text{IO}_{\text{flash}} \ll \text{IO}_{\text{standard}}$.

**What this says:** FlashAttention trades the cost of writing a huge matrix for the cost of doing slightly more arithmetic on smaller tiles. Since we are memory-bound, this is almost always a win.

## 4. Let's Build It — Component by Component

### 4.1 Scaled Dot-Product Attention from Scratch

Let us implement the exact equation from Vaswani et al. using only PyTorch primitives. No `nn.MultiheadAttention` — we want every operation visible.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

torch.manual_seed(42)
np.random.seed(42)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Implements Vaswani et al. Equation (1) from first principles.

    Args:
        Q: Queries,  shape (..., n_queries, d_k)
        K: Keys,     shape (..., n_keys,    d_k)
        V: Values,   shape (..., n_keys,    d_v)
        mask: Optional boolean mask, True = ignore position

    Returns:
        output: shape (..., n_queries, d_v)
        weights: shape (..., n_queries, n_keys)  — for visualization
    """
    d_k = Q.shape[-1]

    # Step 1: raw scores — QK^T
    scores = torch.matmul(Q, K.transpose(-2, -1))   # (..., n_q, n_k)

    # Step 2: scale by 1/sqrt(d_k)
    scores = scores / (d_k ** 0.5)

    # Step 3: apply optional causal mask
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))

    # Step 4: softmax over key dimension
    weights = F.softmax(scores, dim=-1)              # (..., n_q, n_k)

    # Step 5: weighted sum of values
    output = torch.matmul(weights, V)                # (..., n_q, d_v)

    return output, weights


# Quick sanity check
n, d_k, d_v = 6, 32, 32
Q_test = torch.randn(n, d_k)
K_test = torch.randn(n, d_k)
V_test = torch.randn(n, d_v)

out, w = scaled_dot_product_attention(Q_test, K_test, V_test)
print(f"Output shape:  {out.shape}")   # expect (6, 32)
print(f"Weight shape:  {w.shape}")     # expect (6, 6)
print(f"Weight row sum: {w[0].sum():.4f}")  # must be 1.0 (probability distribution)
assert torch.allclose(w.sum(dim=-1), torch.ones(n)), "Weights must sum to 1!"
print("✅ Scaled dot-product attention looks correct.")

### 📊 Visualization Checkpoint — Attention Weights

Let us visualize what the attention matrix actually looks like. We will create a toy sentence and see how attention flows.

In [ ]:
# Create a simple scenario: 8 tokens, model is "copying" attention (keys ≈ queries)
torch.manual_seed(7)
n_tokens = 8
d_model = 16

# Simulate learned projections
token_embeds = torch.randn(n_tokens, d_model)
W_Q = torch.randn(d_model, d_model) * 0.3
W_K = torch.randn(d_model, d_model) * 0.3
W_V = torch.eye(d_model)  # identity — output = attended embeddings

Q_demo = token_embeds @ W_Q
K_demo = token_embeds @ W_K
V_demo = token_embeds @ W_V

_, attn_weights = scaled_dot_product_attention(Q_demo, K_demo, V_demo)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(attn_weights.detach().numpy(), cmap='Blues', vmin=0, vmax=1)
ax.set_title("Attention Weight Matrix\n(entry [i,j] = how much token i attends to token j)",
             fontsize=12, pad=12)
ax.set_xlabel("Key token index  (source of information)", fontsize=11)
ax.set_ylabel("Query token index  (consumer of information)", fontsize=11)
plt.colorbar(im, ax=ax, label="Attention weight")
ax.set_xticks(range(n_tokens))
ax.set_yticks(range(n_tokens))
plt.tight_layout()
plt.savefig("attention_weights.png", dpi=150, bbox_inches='tight')
plt.show()
print("📊 Attention weight matrix visualized.")

### 4.2 Multi-Head Attention

Now we stack multiple attention heads — the full MHA from the Vaswani paper.

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention as in Vaswani et al., Section 3.2.2.

    We keep every projection explicit so the KV cache story is clear:
    each head has its own W_K and W_V. During autoregressive generation,
    ALL of these K and V projections must be stored — that is the KV cache.
    """

    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # One weight matrix per head for Q, K, V (shown explicitly)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def _split_heads(self, x):
        """Reshape (B, T, d_model) → (B, num_heads, T, d_k)"""
        B, T, _ = x.shape
        x = x.view(B, T, self.num_heads, self.d_k)
        return x.transpose(1, 2)   # (B, h, T, d_k)

    def forward(self, x, causal_mask=False):
        B, T, _ = x.shape

        Q = self._split_heads(self.W_Q(x))   # (B, h, T, d_k)
        K = self._split_heads(self.W_K(x))   # (B, h, T, d_k)
        V = self._split_heads(self.W_V(x))   # (B, h, T, d_k)

        mask = None
        if causal_mask:
            # Upper-triangular mask: future tokens cannot be attended to
            mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
            mask = mask.unsqueeze(0).unsqueeze(0)  # broadcast over batch & heads

        out, weights = scaled_dot_product_attention(Q, K, V, mask=mask)

        # Merge heads: (B, h, T, d_k) → (B, T, d_model)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_O(out), weights


# Test MHA
mha = MultiHeadAttention(d_model=64, num_heads=8)
x_in = torch.randn(2, 10, 64)   # batch=2, seq=10, d_model=64
out_mha, w_mha = mha(x_in, causal_mask=True)
print(f"MHA output shape:   {out_mha.shape}")     # (2, 10, 64)
print(f"Attention weights:  {w_mha.shape}")       # (2, 8, 10, 10)

# Count KV parameters per layer — this is what gets cached
kv_params = sum(p.numel() for name, p in mha.named_parameters() if 'W_K' in name or 'W_V' in name)
print(f"\nKV weight params per layer: {kv_params:,}")
print("✅ MultiHeadAttention working correctly.")

### 4.3 Grouped Query Attention (GQA)

Now let us implement GQA — the key contribution of Ainslie et al. The idea is simple but powerful: instead of one K/V head per Q head, we share K/V heads across groups of Q heads.

In [ ]:
class GroupedQueryAttention(nn.Module):
    """
    Grouped Query Attention (Ainslie et al., 2023).

    Key difference from MHA:
      - Q heads:   num_heads
      - KV heads:  num_kv_heads  (= num_heads / num_groups)

    When num_kv_heads == 1          → Multi-Query Attention (MQA)
    When num_kv_heads == num_heads  → standard MHA
    Otherwise                       → GQA
    """

    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int):
        super().__init__()
        assert d_model % num_heads == 0
        assert num_heads % num_kv_heads == 0, "num_heads must be divisible by num_kv_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.num_groups = num_heads // num_kv_heads
        self.d_k = d_model // num_heads

        # Q: full num_heads projections
        self.W_Q = nn.Linear(d_model, num_heads * self.d_k, bias=False)
        # K, V: only num_kv_heads projections  ← this is the saving!
        self.W_K = nn.Linear(d_model, num_kv_heads * self.d_k, bias=False)
        self.W_V = nn.Linear(d_model, num_kv_heads * self.d_k, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, _ = x.shape

        # Project
        Q = self.W_Q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, T, self.num_kv_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.num_kv_heads, self.d_k).transpose(1, 2)

        # Expand K and V so each Q head gets the right KV head
        # K: (B, num_kv_heads, T, d_k) → (B, num_heads, T, d_k)
        K = K.repeat_interleave(self.num_groups, dim=1)
        V = V.repeat_interleave(self.num_groups, dim=1)

        out, weights = scaled_dot_product_attention(Q, K, V)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_O(out), weights


# Compare KV cache sizes
configs = {
    "MHA  (G=1)":  GroupedQueryAttention(64, 8, 8),
    "GQA  (G=4)":  GroupedQueryAttention(64, 8, 2),
    "MQA  (G=8)":  GroupedQueryAttention(64, 8, 1),
}

print(f"{'Config':<18} {'KV head params':>16} {'KV cache factor':>18}")
print("-" * 54)
base_kv = None
for name, model in configs.items():
    kv_params = sum(p.numel() for n, p in model.named_parameters() if 'W_K' in n or 'W_V' in n)
    if base_kv is None:
        base_kv = kv_params
    print(f"{name:<18} {kv_params:>16,}   {kv_params/base_kv:>14.2f}x")

print("\n✅ GQA implemented. Notice how KV params (∝ KV cache size) shrink with more grouping.")

### 📊 Visualization Checkpoint — KV Cache Memory Comparison

Let us make the memory savings concrete across a range of sequence lengths.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left panel: KV cache size vs sequence length ---
ax = axes[0]
seq_lens = np.arange(512, 16385, 512)

d_model = 4096
num_heads = 32
d_k = d_model // num_heads
bytes_per_element = 2  # bfloat16

for label, num_kv in [("MHA (G=1)", 32), ("GQA (G=4)", 8), ("GQA (G=8)", 4), ("MQA (G=32)", 1)]:
    # KV cache = 2 (K+V) × num_kv_heads × seq_len × d_k × bytes
    cache_gb = 2 * num_kv * seq_lens * d_k * bytes_per_element / 1e9
    ax.plot(seq_lens / 1000, cache_gb, linewidth=2.5, label=label, marker='o', markersize=3)

ax.set_xlabel("Sequence Length (k tokens)", fontsize=12)
ax.set_ylabel("KV Cache Size per Layer (GB)", fontsize=12)
ax.set_title("KV Cache Memory: MHA vs GQA vs MQA\n(single layer, d_model=4096, 32 heads, bfloat16)", fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.5, 16.5)

# --- Right panel: Reduction ratios bar chart ---
ax2 = axes[1]
configs_bar = {
    "MHA\n(all heads)": 32,
    "GQA\n(8 KV heads)": 8,
    "GQA\n(4 KV heads)": 4,
    "MQA\n(1 KV head)": 1,
}
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']
names = list(configs_bar.keys())
kv_heads_list = list(configs_bar.values())
reductions = [32 / k for k in kv_heads_list]

bars = ax2.bar(names, reductions, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
for bar, red in zip(bars, reductions):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{red:.0f}×", ha='center', va='bottom', fontsize=13, fontweight='bold')

ax2.set_ylabel("KV Cache Reduction vs MHA", fontsize=12)
ax2.set_title("Memory Reduction Factor\n(relative to standard MHA)", fontsize=11)
ax2.set_ylim(0, 40)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("kv_cache_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("📊 KV cache memory comparison plotted.")

## 5. 🔧 Your Turn

Time to test your understanding. We have three TODOs — one per paper.

### TODO 1: Causal Attention Mask (Vaswani et al.)

In autoregressive generation, token $i$ must not attend to token $j > i$. Implement the mask generation function.

In [ ]:
def make_causal_mask(seq_len: int, device='cpu') -> torch.Tensor:
    """
    Creates a causal (autoregressive) boolean mask.

    Returns a (seq_len, seq_len) tensor where:
      - mask[i, j] = True  if j > i  (future — should be MASKED OUT)
      - mask[i, j] = False if j <= i (past/present — allowed)

    Hint: torch.triu with diagonal=1 gives the upper triangle.
    """
    # ============ TODO ============
    # Step 1: Create a (seq_len, seq_len) tensor of ones (torch.ones)
    # Step 2: Apply torch.triu(..., diagonal=1) to keep only upper triangle
    # Step 3: Convert to bool and move to device
    # ============ END TODO ========
    mask = ???  # YOUR CODE HERE
    return mask

In [ ]:
# ✅ Verification cell for TODO 1
try:
    mask = make_causal_mask(5)
    assert mask.shape == (5, 5), f"Expected (5,5), got {mask.shape}"
    assert mask.dtype == torch.bool, "Mask must be bool"
    # Token 0 should see only itself → top row is [F, T, T, T, T]
    assert not mask[0, 0], "Position (0,0) should NOT be masked (present token)"
    assert mask[0, 1],     "Position (0,1) SHOULD be masked (future token)"
    # Token 4 (last) should see all → bottom row is all False
    assert not mask[4].any(), "Last token should attend to all previous tokens"

    # Visualize the mask
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(mask.numpy(), cmap='Reds', vmin=0, vmax=1)
    ax.set_title("Causal Mask\n(red = masked / blocked future positions)", fontsize=11)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    for i in range(5):
        for j in range(5):
            ax.text(j, i, 'X' if mask[i,j] else '✓', ha='center', va='center',
                    color='white' if mask[i,j] else 'green', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print("✅ Causal mask is correct!")
except NameError:
    print("⚠️  Complete the TODO above first.")
except AssertionError as e:
    print(f"❌ Assertion failed: {e}")

### TODO 2: GQA Cache Size Calculator (Ainslie et al.)

Implement the formula for KV cache size in bytes, supporting MHA, GQA, and MQA.

In [ ]:
def kv_cache_bytes(
    num_layers: int,
    num_heads: int,
    num_kv_heads: int,
    seq_len: int,
    d_head: int,
    bytes_per_element: int = 2
) -> int:
    """
    Computes the total KV cache size in bytes for a full transformer.

    Formula (Ainslie et al., Sec 2):
      size = 2 × num_layers × num_kv_heads × seq_len × d_head × bytes_per_element

    The factor of 2 accounts for both K and V tensors.

    Args:
        num_layers:         number of transformer layers
        num_heads:          total query heads per layer (unused in formula but good to document)
        num_kv_heads:       number of KV heads per layer (= num_heads for MHA, 1 for MQA)
        seq_len:            context length in tokens
        d_head:             dimension of each head (d_model / num_heads)
        bytes_per_element:  2 for bfloat16, 4 for float32

    Returns:
        Total KV cache size in bytes (int)
    """
    # ============ TODO ============
    # Step 1: Multiply out the formula above
    # Step 2: Return as an integer
    # ============ END TODO ========
    size = ???  # YOUR CODE HERE
    return int(size)

In [ ]:
# ✅ Verification cell for TODO 2
try:
    # LLaMA-2 70B approximate config: 80 layers, 64 Q heads, 8 KV heads (GQA), d_head=128
    result_gqa = kv_cache_bytes(
        num_layers=80, num_heads=64, num_kv_heads=8,
        seq_len=4096, d_head=128, bytes_per_element=2
    )
    result_mha = kv_cache_bytes(
        num_layers=80, num_heads=64, num_kv_heads=64,
        seq_len=4096, d_head=128, bytes_per_element=2
    )

    assert result_gqa > 0, "Result must be positive"
    assert result_mha == 8 * result_gqa, \
        f"MHA should be 8x larger than GQA (64 KV heads vs 8). Got ratio {result_mha/result_gqa:.2f}"

    print(f"LLaMA-style 70B model, 4096-token context:")
    print(f"  MHA  KV cache: {result_mha / 1e9:.2f} GB")
    print(f"  GQA  KV cache: {result_gqa / 1e9:.2f} GB  (8 KV heads)")
    print(f"  Savings: {(result_mha - result_gqa)/1e9:.2f} GB  ({(1 - result_gqa/result_mha)*100:.0f}% reduction)")
    print("✅ KV cache calculator is correct!")
except NameError:
    print("⚠️  Complete the TODO above first.")
except AssertionError as e:
    print(f"❌ Assertion failed: {e}")

### TODO 3: IO Cost Estimator (Dao et al., FlashAttention)

Implement the IO cost model that motivates FlashAttention.

In [ ]:
def attention_io_cost(
    seq_len: int,
    d_model: int,
    sram_size_bytes: int = 20 * 1024 * 1024,  # 20 MB — typical A100 SRAM
    bytes_per_element: int = 2
) -> dict:
    """
    Estimates HBM (VRAM) read/write operations for standard vs FlashAttention.

    Standard attention IO (Dao et al., Sec 2.2, Theorem 1):
      Reads:  Q (N×d) + K (N×d) + V (N×d)   = 3Nd elements
      Writes: S=QK^T (N×N) + P=softmax(S) (N×N) + O (N×d)
      Dominant term: O(N²)

    FlashAttention IO (Dao et al., Theorem 2):
      Does NOT write the N×N matrix.
      Uses tiling with block size B ≈ sqrt(sram_size / (4d))
      IO = O(N² × d / sram_size)

    Args:
        seq_len:           N, the sequence length
        d_model:           model dimension d
        sram_size_bytes:   on-chip SRAM capacity in bytes
        bytes_per_element: bytes per scalar

    Returns:
        dict with keys 'standard_io_gb' and 'flash_io_gb'
    """
    N = seq_len
    d = d_model

    # ============ TODO ============
    # Standard attention:
    # Step 1: IO for reading Q, K, V = 3 * N * d * bytes_per_element
    # Step 2: IO for writing S (NxN), P (NxN) = 2 * N * N * bytes_per_element
    # Step 3: IO for writing output O = N * d * bytes_per_element
    # Step 4: standard_io = sum of steps 1, 2, 3

    # FlashAttention:
    # Step 5: flash_io ≈ (N^2 * d * bytes_per_element) / (sram_size_bytes / bytes_per_element)
    # (This captures the tiling savings — see Dao et al. Theorem 2)
    # ============ END TODO ========

    standard_io = ???   # YOUR CODE HERE
    flash_io    = ???   # YOUR CODE HERE

    return {
        'standard_io_gb': standard_io / 1e9,
        'flash_io_gb':    flash_io    / 1e9,
    }

In [ ]:
# ✅ Verification cell for TODO 3
try:
    result_short = attention_io_cost(seq_len=512,  d_model=512)
    result_long  = attention_io_cost(seq_len=4096, d_model=512)

    assert result_short['standard_io_gb'] > 0
    assert result_short['flash_io_gb']    > 0
    # Standard should grow faster (N²) than flash for increasing N
    ratio_standard = result_long['standard_io_gb'] / result_short['standard_io_gb']
    ratio_flash    = result_long['flash_io_gb']    / result_short['flash_io_gb']
    assert ratio_standard > ratio_flash, \
        "Standard IO should grow faster with N than FlashAttention IO"

    print(f"{'Config':<25} {'Standard IO':>15} {'Flash IO':>15} {'Speedup':>10}")
    print("-" * 67)
    for N in [512, 1024, 2048, 4096, 8192]:
        r = attention_io_cost(seq_len=N, d_model=512)
        speedup = r['standard_io_gb'] / r['flash_io_gb']
        print(f"N={N:<21} {r['standard_io_gb']:>12.3f} GB {r['flash_io_gb']:>12.3f} GB {speedup:>8.1f}×")

    print("\n✅ FlashAttention IO model is correct — savings grow with sequence length!")
except NameError:
    print("⚠️  Complete the TODO above first.")
except AssertionError as e:
    print(f"❌ Assertion failed: {e}")

## 6. Putting It All Together

Now we combine everything into one comparison that shows all three papers working in concert.

In [ ]:
def run_full_comparison(
    d_model: int = 128,
    num_heads: int = 8,
    batch_size: int = 2,
    seq_len: int = 64
):
    """
    Runs MHA, GQA (2 groups), and MQA forward passes and compares:
    - Output quality (relative to MHA as reference)
    - KV cache parameter count (proxy for memory use)
    - Parameter efficiency
    """
    torch.manual_seed(42)
    x = torch.randn(batch_size, seq_len, d_model)

    configs = {
        "MHA  (G=1,  8 KV heads)": GroupedQueryAttention(d_model, num_heads, num_heads),
        "GQA  (G=4,  2 KV heads)": GroupedQueryAttention(d_model, num_heads, 2),
        "GQA  (G=8,  1 KV head) ": GroupedQueryAttention(d_model, num_heads, 1),
    }

    results = {}
    ref_out = None

    for name, model in configs.items():
        with torch.no_grad():
            out, weights = model(x)

        kv_params = sum(p.numel() for n, p in model.named_parameters()
                        if 'W_K' in n or 'W_V' in n)
        total_params = sum(p.numel() for p in model.parameters())

        if ref_out is None:
            ref_out = out.clone()
            cos_sim = 1.0
        else:
            # Cosine similarity between outputs (rough quality proxy — random init so not meaningful,
            # but demonstrates the measurement methodology used in the GQA paper)
            cos_sim = F.cosine_similarity(out.flatten(), ref_out.flatten(), dim=0).item()

        results[name] = {
            'kv_params': kv_params,
            'total_params': total_params,
            'output_shape': out.shape,
            'cos_sim_to_mha': cos_sim,
        }

    print(f"{'Configuration':<32} {'KV params':>12} {'Total params':>14} {'% KV':>8}")
    print("-" * 70)
    for name, r in results.items():
        pct_kv = 100 * r['kv_params'] / r['total_params']
        print(f"{name:<32} {r['kv_params']:>12,} {r['total_params']:>14,} {pct_kv:>7.1f}%")

    return results

results = run_full_comparison()
print("\n✅ Full comparison complete.")

### 📊 Comprehensive Visual Summary

In [ ]:
fig = plt.figure(figsize=(15, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ---- Panel 1: IO Cost scaling (FlashAttention argument) ----
ax1 = fig.add_subplot(gs[0, 0])
seq_lens_io = np.array([128, 256, 512, 1024, 2048, 4096, 8192])
standard_ios, flash_ios = [], []
for N in seq_lens_io:
    r = attention_io_cost(N, d_model=512)
    standard_ios.append(r['standard_io_gb'])
    flash_ios.append(r['flash_io_gb'])

ax1.semilogy(seq_lens_io, standard_ios, 'r-o', label='Standard Attention', linewidth=2)
ax1.semilogy(seq_lens_io, flash_ios,    'b-s', label='FlashAttention',      linewidth=2)
ax1.fill_between(seq_lens_io, flash_ios, standard_ios, alpha=0.15, color='green',
                 label='IO saved by Flash')
ax1.set_xlabel("Sequence Length", fontsize=10)
ax1.set_ylabel("HBM IO (GB, log scale)", fontsize=10)
ax1.set_title("Dao et al. 2022\nFlashAttention IO Savings", fontsize=11, fontweight='bold')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# ---- Panel 2: KV cache memory vs seq len for different GQA configs ----
ax2 = fig.add_subplot(gs[0, 1])
seq_lens_kv = np.arange(512, 8193, 256)
gqa_configs = [(32, "MHA (32 KV)", '#d62728'), (8, "GQA (8 KV)", '#ff7f0e'),
               (4, "GQA (4 KV)", '#2ca02c'), (1, "MQA (1 KV)", '#1f77b4')]
for nkv, label, color in gqa_configs:
    cache = [kv_cache_bytes(32, 32, nkv, sl, 128) / 1e9 for sl in seq_lens_kv]
    ax2.plot(seq_lens_kv / 1000, cache, color=color, linewidth=2, label=label)
ax2.set_xlabel("Sequence Length (k tokens)", fontsize=10)
ax2.set_ylabel("KV Cache (GB, 32 layers)", fontsize=10)
ax2.set_title("Ainslie et al. 2023\nGQA Memory Savings", fontsize=11, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ---- Panel 3: Head structure diagram ----
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_xlim(0, 10)
ax3.set_ylim(0, 10)
ax3.axis('off')
ax3.set_title("Head Structure Comparison\n(8 Q-heads shown)", fontsize=11, fontweight='bold')

configs_diagram = [
    ("MHA",  list(range(8)), list(range(8))),
    ("GQA",  list(range(8)), [0,0,0,0,4,4,4,4]),
    ("MQA",  list(range(8)), [0]*8),
]
colors_q  = plt.cm.tab10(np.linspace(0, 0.8, 8))
y_starts  = [8.5, 5.5, 2.5]

for (cfg_name, q_heads, kv_map), y0 in zip(configs_diagram, y_starts):
    ax3.text(0.2, y0, cfg_name, fontsize=9, fontweight='bold', va='center')
    for i, (qi, kvi) in enumerate(zip(q_heads, kv_map)):
        xq = 1.5 + i * 1.0
        # Q head (circle)
        ax3.add_patch(plt.Circle((xq, y0 + 0.4), 0.25, color=colors_q[qi], alpha=0.9))
        ax3.text(xq, y0 + 0.4, 'Q', ha='center', va='center', fontsize=6, color='white', fontweight='bold')
        # KV head (square below)
        kv_x = 1.5 + kvi * 1.0
        ax3.add_patch(plt.Rectangle((kv_x-0.28, y0-0.55), 0.56, 0.4,
                                     color=colors_q[kvi], alpha=0.5))
        ax3.text(kv_x, y0-0.35, 'KV', ha='center', va='center', fontsize=5, fontweight='bold')
        # Arrow
        ax3.annotate('', xy=(kv_x, y0-0.15), xytext=(xq, y0+0.15),
                     arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

# ---- Panel 4: Roofline with GQA effect ----
ax4 = fig.add_subplot(gs[1, :2])
intensity = np.logspace(-1, 3, 300)
peak_compute   = 312.0   # A100 TFLOPS
peak_bandwidth = 2.0     # TB/s → scales: TFLOPS / (TB/s) = ops/byte ridge point

roofline = np.minimum(peak_bandwidth * intensity, peak_compute)
ax4.loglog(intensity, roofline, 'k-', linewidth=2.5, label='Hardware Roofline (A100)')

# Annotate operating points
ops_per_token = lambda n_kv, seq: 2 * n_kv * seq * 128   # attention FLOPs proxy
bytes_per_token = lambda n_kv: 2 * n_kv * 128 * 2        # KV cache bytes loaded

for nkv, label, color, marker in [(32,'MHA','red','o'), (8,'GQA-8','orange','s'),
                                   (1,'MQA','blue','^')]:
    seq = 2048
    ops   = ops_per_token(nkv, seq)
    bw    = bytes_per_token(nkv)
    arith = ops / bw
    perf  = min(peak_bandwidth * arith, peak_compute)
    ax4.scatter([arith], [perf], s=120, color=color, marker=marker, zorder=5, label=label)
    ax4.annotate(label, (arith, perf), textcoords="offset points",
                 xytext=(5, 8), fontsize=9, color=color)

ax4.axvline(peak_compute / peak_bandwidth, color='gray', linestyle='--', alpha=0.6, label='Ridge point')
ax4.set_xlabel("Arithmetic Intensity (FLOPs/byte)", fontsize=11)
ax4.set_ylabel("Achieved TFLOPS", fontsize=11)
ax4.set_title("Roofline Model: GQA Shifts the Operating Point Rightward\n"
              "(higher arithmetic intensity = better hardware utilization)", fontsize=11, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3, which='both')
ax4.fill_between(intensity[intensity < peak_compute/peak_bandwidth],
                 roofline[intensity < peak_compute/peak_bandwidth],
                 alpha=0.08, color='red', label='Memory-bound region')
ax4.text(0.15, 5, "Memory-Bound\n(KV cache traffic\ndominates here)",
         fontsize=9, color='red', alpha=0.8)
ax4.text(80, 50, "Compute-Bound", fontsize=9, color='green', alpha=0.8)

# ---- Panel 5: Summary text ----
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
summary = (
    "Key Takeaways\n"
    "─────────────────────────────\n\n"
    "📄 Vaswani et al. 2017\n"
    "  Scaled dot-product attention\n"
    "  QK^T/√d_k grounds the KV\n"
    "  cache — K and V tensors\n"
    "  are exactly what gets stored.\n\n"
    "📄 Ainslie et al. 2023\n"
    "  GQA reduces KV heads\n"
    "  from h to h/G, slashing\n"
    "  cache by factor G.\n"
    "  Shifts roofline point right.\n\n"
    "📄 Dao et al. 2022\n"
    "  FlashAttention avoids\n"
    "  writing N×N to VRAM.\n"
    "  IO drops from O(N²)\n"
    "  to O(N²d/M).\n\n"
    "Together: less memory,\n"
    "faster, more scalable."
)
ax5.text(0.05, 0.98, summary, transform=ax5.transAxes,
         fontsize=9.5, va='top', fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#f8f9fa', edgecolor='#dee2e6'))

plt.suptitle("Further Reading: Three Papers That Define Modern LLM Inference",
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig("further_reading_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print("📊 Comprehensive summary visualization complete.")

## 7. Training and Results

Let us run a small but complete experiment: we will train a tiny two-layer transformer with MHA and with GQA side by side on a character-level language modeling task, and observe that GQA reaches comparable loss with a significantly smaller KV footprint.

In [ ]:
# Tiny transformer training experiment
import time

# ── Tiny dataset: repeating pattern (easy to learn, quick to train) ──
torch.manual_seed(42)
vocab_size = 16
seq_len    = 32
d_model    = 64
num_heads  = 8
n_layers   = 2
n_steps    = 300
lr         = 3e-3

# Generate synthetic sequential data
def make_batch(batch_size=32, seq_len=32, vocab=16):
    # Pattern: each token predicts next token in a repeating cycle
    starts = torch.randint(0, vocab, (batch_size,))
    seqs   = torch.stack([(starts + i) % vocab for i in range(seq_len + 1)], dim=1)
    return seqs[:, :-1], seqs[:, 1:]  # input, target

class TinyTransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads):
        super().__init__()
        self.attn = GroupedQueryAttention(d_model, num_heads, num_kv_heads)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.attn(self.ln1(x))
        x = x + attn_out
        x = x + self.ff(self.ln2(x))
        return x

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_kv_heads, n_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.Sequential(*[
            TinyTransformerBlock(d_model, num_heads, num_kv_heads)
            for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        x = self.blocks(x)
        return self.head(x)

def train_model(num_kv_heads, label):
    torch.manual_seed(42)
    model = TinyTransformer(vocab_size, d_model, num_heads, num_kv_heads, n_layers)
    optim = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []

    for step in range(n_steps):
        x, y = make_batch()
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
        optim.zero_grad()
        loss.backward()
        optim.step()
        if step % 10 == 0:
            losses.append(loss.item())

    kv_size = kv_cache_bytes(n_layers, num_heads, num_kv_heads, seq_len, d_model // num_heads)
    print(f"  {label:<25} Final loss: {losses[-1]:.4f}   KV cache: {kv_size:,} bytes")
    return losses

print("Training comparison (this takes ~15 seconds)...\n")
t0 = time.time()
losses_mha  = train_model(8, "MHA  (8 KV heads)")
losses_gqa4 = train_model(2, "GQA  (2 KV heads)")
losses_mqa  = train_model(1, "MQA  (1 KV head) ")
print(f"\nTotal training time: {time.time()-t0:.1f}s")
print("✅ Training complete.")

### 📊 Training Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
steps = list(range(0, n_steps, 10))

ax.plot(steps, losses_mha,  'r-o',  markersize=4, linewidth=2,   label='MHA  (8 KV heads)')
ax.plot(steps, losses_gqa4, 'g-s',  markersize=4, linewidth=2,   label='GQA  (2 KV heads, G=4)')
ax.plot(steps, losses_mqa,  'b-^',  markersize=4, linewidth=2,   label='MQA  (1 KV head,  G=8)')

ax.set_xlabel("Training Step", fontsize=12)
ax.set_ylabel("Cross-Entropy Loss", fontsize=12)
ax.set_title("Training Loss: MHA vs GQA vs MQA\n"
             "(GQA recovers most MHA quality at a fraction of the KV memory cost)",
             fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Annotate final losses
for losses, color, label in [(losses_mha,'red','MHA'), (losses_gqa4,'green','GQA'), (losses_mqa,'blue','MQA')]:
    ax.annotate(f'{losses[-1]:.3f}', xy=(steps[-1], losses[-1]),
                xytext=(5, 0), textcoords='offset points',
                fontsize=9, color=color, fontweight='bold', va='center')

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. 🎯 Final Output

Let us produce one final, screenshot-worthy visualization that synthesizes the entire story of this series — the papers, the concepts, and the memory/compute trade-offs — in a single poster.

In [ ]:
fig = plt.figure(figsize=(16, 11))
fig.patch.set_facecolor('#0d1117')

# ── Title ──
fig.text(0.5, 0.965, "The Three Papers Behind Modern LLM Inference",
         ha='center', va='top', fontsize=18, color='white', fontweight='bold')
fig.text(0.5, 0.940, "Vaswani et al. 2017  •  Dao et al. 2022  •  Ainslie et al. 2023",
         ha='center', va='top', fontsize=11, color='#8b949e')

# ── Set up grid ──
gs = GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.3,
              top=0.90, bottom=0.07, left=0.06, right=0.97)
panel_bg = '#161b22'
text_color = '#e6edf3'
accent = '#58a6ff'

# ─── Panel 1: Attention equation visualization ───
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(panel_bg)
ax1.tick_params(colors=text_color)
for spine in ax1.spines.values(): spine.set_color('#30363d')

torch.manual_seed(0)
n_vis = 8
q = torch.randn(n_vis, 16)
k = torch.randn(n_vis, 16)
v = torch.eye(n_vis)[:, :n_vis]
_, w = scaled_dot_product_attention(q, k, v.float())
im = ax1.imshow(w.detach().numpy(), cmap='Blues', aspect='auto')
ax1.set_title("Vaswani et al. (2017)\nAttention Weight Matrix  softmax(QKᵀ/√d_k)V",
              color=text_color, fontsize=9, pad=6)
ax1.set_xlabel("Key position", color=text_color, fontsize=8)
ax1.set_ylabel("Query position", color=text_color, fontsize=8)
plt.colorbar(im, ax=ax1).ax.yaxis.set_tick_params(color=text_color)

# ─── Panel 2: IO cost comparison ───
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor(panel_bg)
for spine in ax2.spines.values(): spine.set_color('#30363d')
ax2.tick_params(colors=text_color)

seq_range = np.array([256, 512, 1024, 2048, 4096, 8192])
std_io, fl_io = [], []
for N in seq_range:
    r = attention_io_cost(N, 512)
    std_io.append(r['standard_io_gb'])
    fl_io.append(r['flash_io_gb'])

ax2.semilogy(seq_range, std_io, color='#f85149', linewidth=2.5, marker='o', markersize=5,
             label='Standard Attention')
ax2.semilogy(seq_range, fl_io,  color='#3fb950', linewidth=2.5, marker='s', markersize=5,
             label='FlashAttention')
ax2.fill_between(seq_range, fl_io, std_io, alpha=0.12, color='#3fb950')
ax2.set_title("Dao et al. (2022)\nHBM IO: Standard vs FlashAttention",
              color=text_color, fontsize=9, pad=6)
ax2.set_xlabel("Sequence Length", color=text_color, fontsize=8)
ax2.set_ylabel("HBM IO (GB, log scale)", color=text_color, fontsize=8)
ax2.legend(fontsize=8, facecolor='#21262d', labelcolor=text_color)
ax2.grid(True, alpha=0.2, color='#30363d')

# ─── Panel 3: GQA memory reduction ───
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor(panel_bg)
for spine in ax3.spines.values(): spine.set_color('#30363d')
ax3.tick_params(colors=text_color)

seq_kv = np.arange(512, 8193, 256)
palette = ['#f85149', '#d29922', '#3fb950', '#58a6ff']
for (nkv, lbl), col in zip([(32,"MHA"),(8,"GQA-8"),(4,"GQA-4"),(1,"MQA")], palette):
    mem = [kv_cache_bytes(32, 32, nkv, sl, 128) / 1e9 for sl in seq_kv]
    ax3.plot(seq_kv/1000, mem, color=col, linewidth=2, label=lbl)
ax3.set_title("Ainslie et al. (2023)\nKV Cache Size vs Sequence Length",
              color=text_color, fontsize=9, pad=6)
ax3.set_xlabel("Seq. Length (k tokens)", color=text_color, fontsize=8)
ax3.set_ylabel("KV Cache GB (32 layers)", color=text_color, fontsize=8)
ax3.legend(fontsize=8, facecolor='#21262d', labelcolor=text_color)
ax3.grid(True, alpha=0.2, color='#30363d')

# ─── Panel 4: Roofline ───
ax4 = fig.add_subplot(gs[1, :2])
ax4.set_facecolor(panel_bg)
for spine in ax4.spines.values(): spine.set_color('#30363d')
ax4.tick_params(colors=text_color)

intensity = np.logspace(-1, 3, 400)
roof = np.minimum(2.0 * intensity, 312.0)
ax4.loglog(intensity, roof, color=accent, linewidth=3, label='A100 Roofline')
ax4.fill_between(intensity, roof * 0.0, roof, alpha=0.05, color=accent)

# Operating points
for nkv, lbl, col, mk in [(32,'MHA','#f85149','o'),(8,'GQA(G=4)','#d29922','s'),(1,'MQA','#3fb950','^')]:
    ai = (2 * nkv * 2048 * 128) / (2 * nkv * 128 * 2)
    pf = min(2.0 * ai, 312.0)
    ax4.scatter([ai], [pf], s=180, color=col, marker=mk, zorder=6, label=lbl)
    ax4.annotate(f' {lbl}', (ai, pf), fontsize=9, color=col, va='center')

ax4.axvline(312/2, color='#8b949e', linestyle='--', alpha=0.5)
ax4.text(312/2 * 1.1, 2, 'Ridge\nPoint', color='#8b949e', fontsize=8)
ax4.set_xlabel("Arithmetic Intensity (FLOPs / byte)", color=text_color, fontsize=10)
ax4.set_ylabel("Achieved Performance (TFLOPS)", color=text_color, fontsize=10)
ax4.set_title("Roofline Model: GQA & FlashAttention Push Us Toward Compute-Bound",
              color=text_color, fontsize=10, pad=6)
ax4.legend(fontsize=9, facecolor='#21262d', labelcolor=text_color)
ax4.grid(True, alpha=0.15, color='#30363d', which='both')

# ─── Panel 5: Training curves (dark theme) ───
ax5 = fig.add_subplot(gs[1, 2])
ax5.set_facecolor(panel_bg)
for spine in ax5.spines.values(): spine.set_color('#30363d')
ax5.tick_params(colors=text_color)

step_range = list(range(0, n_steps, 10))
ax5.plot(step_range, losses_mha,  color='#f85149', linewidth=2,   label='MHA')
ax5.plot(step_range, losses_gqa4, color='#3fb950', linewidth=2,   label='GQA (G=4)')
ax5.plot(step_range, losses_mqa,  color='#58a6ff', linewidth=2,   label='MQA (G=8)')
ax5.set_title("Training Loss\nMHA vs GQA vs MQA", color=text_color, fontsize=9, pad=6)
ax5.set_xlabel("Training Step", color=text_color, fontsize=8)
ax5.set_ylabel("Cross-Entropy Loss", color=text_color, fontsize=8)
ax5.legend(fontsize=8, facecolor='#21262d', labelcolor=text_color)
ax5.grid(True, alpha=0.2, color='#30363d')

# ── Footer ──
fig.text(0.5, 0.01,
         "Vizuara — The Good and Evil of the Key-Value Cache  |  vizuara.ai",
         ha='center', fontsize=9, color='#8b949e')

plt.savefig("kv_cache_further_reading_poster.png", dpi=180, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("🎉 Poster saved as 'kv_cache_further_reading_poster.png'")
print("    Screenshot this — you have earned it!")

## 9. Reflection and Next Steps

### 🤔 Reflection Questions

Let us revisit the question we posed at the start of Section 2, plus a few more to deepen your understanding.

**1. The GQA arithmetic intensity question** *(from the teaser)*

> When GQA reduces KV heads by a factor of $G$, the number of bytes loaded from the KV cache per generation step drops by $G\times$. But the number of attention FLOPs does *not* drop (we still have the same number of Q heads). So arithmetic intensity = FLOPs / bytes **increases** by $G\times$. This moves the operating point **rightward** on the roofline — closer to the compute-bound ridge. That is a win: the hardware is better utilized.

**2. Why does FlashAttention not change the mathematical result?**

> FlashAttention is an *exact* algorithm — it computes the same output as standard attention, to floating-point precision. The trick is purely about *ordering* operations and using on-chip SRAM as a scratchpad, not about approximating. If you run both on the same input you get identical outputs (modulo floating-point associativity rounding).

**3. When would GQA hurt quality more — on a small model or a large model?**

> On small models. With fewer total heads, each head carries a larger share of the representational burden. Sharing KV heads removes diversity more aggressively. On large models (many heads, large $d_{\text{model}}$), the GQA paper shows the quality gap essentially vanishes after fine-tuning.

**4. Why is the factor of $\sqrt{d_k}$ in attention important for stability?**

> As $d_k$ grows, the dot products $q \cdot k$ grow in magnitude (their variance scales with $d_k$ for unit-normal inputs). Large magnitudes push softmax into saturation — gradients become near-zero and learning stalls. Dividing by $\sqrt{d_k}$ keeps the variance constant regardless of head dimension.

### 🏆 Optional Challenges

If you want to go further, here are three challenges ordered by difficulty:

---

**⭐ Challenge 1 — Implement KV Caching**

Extend `GroupedQueryAttention` to support a `past_kv` argument. On each forward pass, if `past_kv` is provided, concatenate the new K and V with the cached ones along the sequence dimension and return the updated cache. This is exactly how inference servers work.

In [ ]:
def forward_with_cache(self, x, past_kv=None):
    # Your implementation here
    # Hint: K_new = cat([past_kv['K'], K_current], dim=-2)
    # Return: output, {'K': K_full, 'V': V_full}
    pass

---

**⭐⭐ Challenge 2 — Profile the Memory Bandwidth**

Use `torch.cuda.memory_allocated()` before and after a GQA forward pass for different `num_kv_heads` values. Plot actual GPU memory usage versus our theoretical formula. Do they match? What overheads does PyTorch add?

---

**⭐⭐⭐ Challenge 3 — Implement Tiled Attention (Mini FlashAttention)**

Implement a tiled version of `scaled_dot_product_attention` that processes the key/value sequence in blocks of size `B` and accumulates the output using the online softmax trick (log-sum-exp running maximum). Verify it gives the same output as the naive version. See Dao et al. Algorithm 1 for the pseudocode.

---

### 📚 The Papers — Where to Find Them

| Paper | Link |
|---|---|
| Vaswani et al. 2017 — *Attention Is All You Need* | [arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762) |
| Dao et al. 2022 — *FlashAttention* | [arxiv.org/abs/2205.14135](https://arxiv.org/abs/2205.14135) |
| Ainslie et al. 2023 — *GQA* | [arxiv.org/abs/2305.13245](https://arxiv.org/abs/2305.13245) |

A practical reading order: Vaswani first (for the foundation), then Dao (for the memory argument you now understand deeply), then Ainslie (for how production systems actually deploy it). You have all the vocabulary you need. Go read the originals — you will be surprised how much you already understand.

---

*Thank you for working through the Vizuara KV Cache series. The concepts here — roofline model, arithmetic intensity, IO-awareness, grouped attention — are not just useful for understanding existing systems. They are the vocabulary that engineers use to design the next ones. Keep building.*